# recipe-dataclass — ex3: clamp_forward — Recipe with 1 parent and 2 non-empty kwargs

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `recipe-dataclass`. Running the final beacon cell reports progress against the `Backprop: Recipe dataclass` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Recipe dataclass` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`recipe-dataclass`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "recipe-dataclass"
DD_SUBTOPIC = "Backprop: Recipe dataclass"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Recipe for `clamp(x, min=, max=)` — 1 parent, 2 kwargs

Ex1 (log: 1-arg) and ex2 (add: 2-arg) had `kwargs == {}`. The deepening move populates `recipe.kwargs` for the first time: `clamp_forward(x, min=lo, max=hi)` has ONE Tensor parent but two scalar kwargs that the back fn needs to clip at reverse time.

```python
@dataclass
class Recipe:
    func: Callable
    args: tuple    # → (x.array,)   one positional Tensor
    kwargs: dict   # → {'min': -1.0, 'max': 1.0}   non-empty!
    parents: dict  # → {0: x}        only positional parent
```

**Why the kwargs DON'T appear in parents.** `min` and `max` are scalar floats — non-Tensor inputs are filtered out of `parents` (same rule as ex1's parents-dict drill). They live ONLY in `recipe.kwargs` because the back fn needs them, but they have no gradient flow (you can't take d/dmin of a clamp output).

**The reverse pass replay.** `clamp_back(grad_out, out, x, min=..., max=...)` masks the gradient where `x` was clipped: `grad_in = grad_out * ((x >= min) & (x <= max)).float()`. The mask is computed from `recipe.kwargs` — without them stored, the back fn would have no way to know where the clipping happened.

### Exercise 3 — clamp_forward — Recipe with 1 parent and 2 non-empty kwargs

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the 4-field Recipe construction to `clamp_forward(x, min, max)` where the Recipe has ONE Tensor parent and TWO non-Tensor kwargs (min, max) — distinguishing what goes into parents vs what goes into kwargs.
> Keywords: recipe, clamp, kwargs-non-empty, scalar-kwarg
> ```

**KCs targeted:** `recipe-dataclass`, `recipe-kwargs-stores-non-tensor-args`

Implement `ex3_clamp_forward(x, *, min_val, max_val)` for `MiniTensor` input `x` (the `min_val` / `max_val` naming avoids shadowing Python builtins inside the implementation).

Required behaviour:

1. Compute `out_raw = x.array.clamp(min=min_val, max=max_val)`.
2. Return a NEW `MiniTensor` with `.array = out_raw` and a fully populated `.recipe` such that:
   - `recipe.func` is `t.clamp` (the forward op).
   - `recipe.args` is `(x.array,)` — one-tuple of the unboxed Tensor input.
   - `recipe.kwargs` is `{'min': min_val, 'max': max_val}` — EXACT keys `'min'` and `'max'` (not `'min_val'`), and the SAME numeric values that were passed in.
   - `recipe.parents` is `{0: x}` — ONLY the positional Tensor parent. The `min_val` and `max_val` floats DO NOT go into parents (non-Tensor → no grad flow).

The MiniTensor class and Recipe dataclass are pre-defined in the test cell — you just construct and return.

Why the kwargs naming flip matters: at backward time, `clamp_back(grad_out, out, x, **recipe.kwargs)` must receive `min=..., max=...` (matching `torch.clamp`'s own signature). Storing under `'min_val'`/`'max_val'` would break the splat.

In [ ]:
def ex3_clamp_forward(x, *, min_val, max_val):
    out_raw = x.array.clamp(min=min_val, max=max_val)
    recipe = Recipe(
        func=t.clamp,
        args=(x.array,),
        kwargs={'min': min_val, 'max': max_val},
        parents={0: x},
    )
    out = MiniTensor(out_raw, recipe=recipe)
    return out


<details><summary>Solution</summary>

```python
def ex3_clamp_forward(x, *, min_val, max_val):
    out_raw = x.array.clamp(min=min_val, max=max_val)
    recipe = Recipe(
        func=t.clamp,
        args=(x.array,),
        kwargs={'min': min_val, 'max': max_val},
        parents={0: x},
    )
    out = MiniTensor(out_raw, recipe=recipe)
    return out
```

**`'min'` not `'min_val'` for kwargs keys.** The naming is for the Python function signature (avoiding the `min` builtin); the stored keys must match what `torch.clamp` accepts as kwargs. Splat at backward time: `t.clamp(..., **recipe.kwargs)` would fail with 'unexpected keyword argument min_val' otherwise.

**Floats stay in kwargs, never in parents.** `min_val` and `max_val` are scalars — `isinstance(x, MiniTensor)` is False — so they're filtered out of parents (same rule as ex1's parents drill). They appear ONLY in `recipe.kwargs` because the back fn needs them to know where the clipping happened.

**`parents[0] is x` — boxed, not unboxed.** `recipe.args[0]` is `x.array` (unboxed for the forward call). `recipe.parents[0]` is the original boxed `MiniTensor` (so the reverse pass can walk back to its own recipe and continue the chain). This dual storage is intentional.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()